In [3]:
%env DB_PASSWORD=5J8DhII0RRsPW1

env: DB_PASSWORD=5J8DhII0RRsPW1


In [5]:
import pandas as pd
import re
from constants.db_connections import ENGINE_READ_ONLY
import os
import paramiko


pd.set_option('display.float_format', '{:.2f}'.format)  # Adjust the format to the desired decimal places

In [6]:

# Set your server details
hostname = 'dandyweb01fl'  # Replace with your server's IP or hostname
port = 22                       # Usually 22 for SSH
username = 'glj523'      # Replace with your username
password = 'Wtcantfw36c!123'      # Replace with your password

remote_directories = ['/datasets/caeg_fastq/2023/20230927_A00706_0773_AHNLW5DSX5_AOZCK/eDNALib060-Kurt',
                      '/datasets/caeg_fastq/2023/20230927_A00706_0773_AHNLW5DSX5_AOZCK/eDNALib060-Thorfinn',
                      '/datasets/caeg_fastq/2024/20240702_A00706_0862_BH5F5KDSX7_WBDQ4_new/ssDNALib0019']

lib_ids_all = {}

remote_directory = remote_directories[0]  # Replace with the directory you want to list


# Create an SSH client
ssh = paramiko.SSHClient()
ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())

try:
    # Connect to the server
    ssh.connect(hostname, port, username, password)

    # Run the command to list files and directories

    for dir in remote_directories:
    
        stdin, stdout, stderr = ssh.exec_command(f"ls {dir} | grep '\.fastq.*$'")

        _, test, _ = ssh.exec_command(f"ls {dir} | grep '\.fastq.*$' | wc -l")
    

        # Process the output
        file_names = stdout.read().decode().splitlines()
        test = test.read().decode()

        lib_ids = [file_name.split("-")[0] for file_name in file_names]

        if int(test) != len(lib_ids):
            raise Exception("Error")

        lib_ids_all[dir.split("/")[-1]] = list(set(lib_ids))
    

        # Print the first 8 letters of each file/directory name
   

except Exception as e:
    print(f"An error occurred: {e}")

finally:
    # Close the connection
    ssh.close()

df = pd.read_sql(sql='select * from test_1.mega_table_qc_split_mat', con=ENGINE_READ_ONLY)

In [90]:
abi_data = pd.read_csv(r"c:\Users\glj523\Downloads\CambodiaLibs - cambodia_libs.tsv", sep="\t")

In [82]:
kurt = lib_ids_all["eDNALib060-Kurt"]
thorf = lib_ids_all["eDNALib060-Thorfinn"]
sslib = lib_ids_all["ssDNALib0019"]

In [83]:
cols = ['library_id', 
        'qc_type',
        'sample_name',
        'fastqc_raw__Total Sequences',
        'fastqc_trimmed__Total Sequences',
        r'fastqc_raw__%GC',
        r'fastqc_trimmed__%GC',
        'fastp__insert_size',
        'samtools_stats__reads_mapped']

In [94]:
abi_data[abi_data["Library.ID"].isin(thorf)]

,BulkSampleID,ArchiveSampleID,DepthSampledCalTape,Library.ID,Filebase,Basedir,raw_reads,collapsed_reads,assigned_reads,interesting?
6,CAM22-08,LV3005273239,60.50,LV7009026440,LV3005273239_LV7009026440,/projects/caeg/data/production/LV30/0527/LV300...,173483218,125537320,334752,NaN
7,CAM22-08,LV3005273245,62.00,LV7009026452,LV3005273245_LV7009026452,/projects/caeg/data/production/LV30/0527/LV300...,152122683,106577728,233851,NaN
8,CAM22-08,LV3005273244,63.50,LV7009026464,LV3005273244_LV7009026464,/projects/caeg/data/production/LV30/0527/LV300...,128310069,89940373,239494,NaN
9,CAM22-08,LV3005273232,64.50,LV7009026476,LV3005273232_LV7009026476,/projects/caeg/data/production/LV30/0527/LV300...,138992048,100284795,250609,NaN
10,CAM22-08,LV3005273231,66.00,LV7009026488,LV3005273231_LV7009026488,/projects/caeg/data/production/LV30/0527/LV300...,146749411,99747038,395744,High yield
11,CAM22-08,LV3005273243,67.50,LV7009026500,LV3005273243_LV7009026500,/projects/caeg/data/production/LV30/0527/LV300...,132343569,90466790,224841,NaN
12,CAM22-08,LV3005273240,69.00,LV7009026512,LV3005273240_LV7009026512,/projects/caeg/data/production/LV30/0527/LV300...,146637383,106089399,272900,NaN
13,CAM22-08,LV3005273234,70.50,LV7009026524,LV3005273234_LV7009026524,/projects/caeg/data/production/LV30/0527/LV300...,133910098,94691470,250958,NaN
14,CAM22-08,LV3005273228,71.50,LV7009026439,LV3005273228_LV7009026439,/projects/caeg/data/production/LV30/0527/LV300...,107950269,78545899,258398,NaN
15,CAM22-08,LV3005273247,73.00,LV7009026451,LV3005273247_LV7009026451,/projects/caeg/data/production/LV30/0527/LV300...,168139818,120817101,388699,NaN


In [84]:
sslib_qc = df[df["library_id"].isin(sslib)][cols]
thorf_qc = df[df["library_id"].isin(thorf)][cols]
kurt_qc = df[df["library_id"].isin(kurt)][cols]

In [81]:
if sslib_qc["sample_name"].duplicated().sum() != 0:
    raise Exception()


In [61]:
kurt.describe().iloc[0]

fastqc_raw__Total Sequences       0.00
fastqc_trimmed__Total Sequences   0.00
fastqc_raw__%GC                   0.00
fastqc_trimmed__%GC               0.00
samtools_stats__reads_mapped      0.00
Name: count, dtype: float64